# Tarea 2: Análisis Exploratorio de Datos (EDA) y Particionado de Features

Este notebook contiene el Análisis Exploratorio de Datos (EDA) sobre el dataset unificado `matches_clean.csv`. Analizaremos las distribuciones, correlaciones, ventajas de localía por confederación y generaremos los gráficos correspondientes en `docs/eda_plots/`. Por último, particionaremos las features por fechas para evitar el *data leakage* temporal.

In [ ]:
import os
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

PROCESSED_DIR = "../data/processed"
PLOT_DIR = "../docs/eda_plots"
os.makedirs(PLOT_DIR, exist_ok=True)

df = pd.read_csv(os.path.join(PROCESSED_DIR, "matches_clean.csv"))
df['date'] = pd.to_datetime(df['date'])

sns.set_theme(style="whitegrid")
print(f"Dataset cargado con éxito. Total de filas: {len(df)}")

### 1. Distribución de Resultados

Graficamos la distribución del resultado del partido (derrota local, empate, victoria local).

In [ ]:
plt.figure(figsize=(8, 5))
ax = sns.countplot(data=df, x='result', palette='viridis', hue='result', legend=False)
ax.set_title("Distribución de Resultados (Perspectiva Local)")
ax.set_xlabel("Resultado (0=Derrota local, 1=Empate, 2=Victoria local)")
ax.set_ylabel("Cantidad de Partidos")
plt.savefig(os.path.join(PLOT_DIR, "distribucion_resultados.png"), dpi=300, bbox_inches='tight')
plt.show()

### 2. Distribución de Goles por Torneo

Visualizamos la cantidad total de goles anotados por partido en los 8 torneos más frecuentes.

In [ ]:
top_tournaments = df['tournament'].value_counts().head(8).index
df_top_tourn = df[df['tournament'].isin(top_tournaments)].copy()
df_top_tourn['total_goals'] = df_top_tourn['home_score'] + df_top_tourn['away_score']

plt.figure(figsize=(12, 6))
ax = sns.boxplot(data=df_top_tourn, x='tournament', y='total_goals', palette='Set3', hue='tournament', legend=False)
ax.set_title("Distribución de Goles Totales por Torneo (Top 8 Torneos)")
ax.set_xlabel("Torneo")
ax.set_ylabel("Goles por Partido")
plt.xticks(rotation=45, ha='right')
plt.savefig(os.path.join(PLOT_DIR, "goles_por_torneo.png"), dpi=300, bbox_inches='tight')
plt.show()

### 3. Heatmap de Correlaciones

Analizamos la correlación lineal entre las variables numéricas clave del modelo.

In [ ]:
num_cols = [
    'elo_home', 'elo_away', 'elo_diff',
    'home_wins_5', 'home_goals_scored_5',
    'away_wins_5', 'away_goals_scored_5',
    'h2h_home_wins', 'h2h_draws', 'h2h_away_wins',
    'is_neutral', 'tournament_weight', 'phase_encoded', 'result'
]
corr_matrix = df[num_cols].corr()

plt.figure(figsize=(12, 10))
sns.heatmap(corr_matrix, annot=True, fmt=".2f", cmap="coolwarm", cbar=True, square=True)
plt.title("Heatmap de Correlaciones de Variables Numéricas")
plt.savefig(os.path.join(PLOT_DIR, "heatmap_correlaciones.png"), dpi=300, bbox_inches='tight')
plt.show()

### 4. Ventaja de Localía por Confederación

Mapeamos los países a sus confederaciones de fútbol y analizamos la proporción de resultados locales para partidos no neutrales.

In [ ]:
def get_confederation(team):
    team = team.strip()
    if team in ['Argentina', 'Brazil', 'Uruguay', 'Colombia', 'Chile', 'Peru', 'Ecuador', 'Paraguay', 'Bolivia', 'Venezuela']:
        return 'CONMEBOL'
    elif team in ['USA', 'United States', 'Mexico', 'Canada', 'Costa Rica', 'Honduras', 'Panama', 'El Salvador', 'Jamaica', 'Trinidad and Tobago', 'Haiti', 'Guatemala', 'Cuba', 'Martinique', 'Guadeloupe', 'Curaçao', 'Netherlands Antilles', 'Suriname', 'Guyana', 'British Guiana']:
        return 'CONCACAF'
    elif team in ['Germany', 'France', 'Spain', 'Italy', 'England', 'Netherlands', 'Portugal', 'Belgium', 'Croatia', 'Denmark', 'Sweden', 'Switzerland', 'Poland', 'Ukraine', 'Russia', 'Soviet Union', 'CIS', 'Turkey', 'Greece', 'Austria', 'Czechia', 'Czechoslovakia', 'Slovakia', 'Romania', 'Bulgaria', 'Hungary', 'Serbia', 'FR Yugoslavia', 'Serbia and Montenegro', 'Slovenia', 'Bosnia and Herzegovina', 'North Macedonia', 'Macedonia', 'Wales', 'Scotland', 'Northern Ireland', 'Republic of Ireland', 'Ireland', 'Norway', 'Finland', 'Iceland', 'Albania', 'Georgia', 'Armenia', 'Azerbaijan', 'Belarus', 'Cyprus', 'Estonia', 'Faroe Islands', 'Gibraltar', 'Kazakhstan', 'Kosovo', 'Latvia', 'Liechtenstein', 'Lithuania', 'Luxembourg', 'Malta', 'Moldova', 'Montenegro', 'San Marino', 'Andorra']:
        return 'UEFA'
    elif team in ['Egypt', 'Senegal', 'Morocco', 'Algeria', 'Tunisia', 'Nigeria', 'Cameroon', 'Ghana', 'Ivory Coast', 'South Africa', 'Mali', 'Guinea', 'Burkina Faso', 'Upper Volta', 'Angola', 'DR Congo', 'Zaïre', 'Belgian Congo', 'Congo-Léopoldville', 'Congo-Kinshasa', 'Congo', 'Zambia', 'Northern Rhodesia', 'Zimbabwe', 'Southern Rhodesia', 'Kenya', 'Uganda', 'Tanzania', 'Tanganyika', 'Sudan', 'Ethiopia', 'Madagascar', 'Mauritius', 'Seychelles', 'Benin', 'Dahomey', 'Togo', 'Gabon', 'Equatorial Guinea', 'Central African Republic', 'Chad', 'Niger', 'Mauritania', 'Cape Verde', 'Guinea-Bissau', 'Portuguese Guinea', 'Sierra Leone', 'Liberia', 'Gambia', 'Somalia', 'Djibouti', 'French Somaliland', 'Eritrea', 'Rwanda', 'Burundi', 'South Sudan', 'Lesotho', 'Eswatini', 'Swaziland', 'Namibia', 'Botswana', 'Mozambique', 'Malawi', 'Nyasaland', 'Comoros', 'Libya']:
        return 'CAF'
    elif team in ['Japan', 'South Korea', 'Australia', 'Iran', 'Saudi Arabia', 'Iraq', 'UAE', 'China', 'Qatar', 'Syria', 'Uzbekistan', 'Jordan', 'Oman', 'India', 'Bahrain', 'Kuwait', 'Lebanon', 'Vietnam', 'Vietnam Republic', 'Thailand', 'Indonesia', 'Dutch East Indies', 'Malaysia', 'Malaya', 'Singapore', 'Philippines', 'Myanmar', 'Burma', 'North Korea', 'Hong Kong', 'Macau', 'Chinese Taipei', 'Taiwan', 'Guam', 'Mongolia', 'Nepal', 'Bangladesh', 'Sri Lanka', 'Ceylon', 'Pakistan', 'Maldives', 'Bhutan', 'Afghanistan', 'Tajikistan', 'Turkmenistan', 'Kyrgyzstan', 'Yemen', 'Palestine', 'Cambodia', 'Laos', 'Brunei', 'Timor-Leste', 'Israel', 'Mandatory Palestine']:
        return 'AFC'
    elif team in ['New Zealand', 'Fiji', 'Tahiti', 'Solomon Islands', 'Vanuatu', 'New Hebrides', 'New Caledonia', 'Samoa', 'Western Samoa', 'Tonga', 'American Samoa', 'Cook Islands', 'Tuvalu', 'Kiribati', 'Niue', 'Papua New Guinea']:
        return 'OFC'
    else:
        return 'Other'

df['confed_home'] = df['home_team'].apply(get_confederation)
df_home_adv = df[(df['confed_home'] != 'Other') & (df['neutral'] == False)].copy()

grouped = df_home_adv.groupby('confed_home')['result'].value_counts(normalize=True).unstack().fillna(0)
grouped = grouped.rename(columns={2: 'Victoria Local', 1: 'Empate', 0: 'Victoria Visitante'})

plt.figure(figsize=(10, 6))
grouped.plot(kind='bar', stacked=True, colormap='viridis', width=0.6, ax=plt.gca())
plt.title("Rendimiento del Local por Confederación (Partidos No Neutrales)")
plt.xlabel("Confederación del Local")
plt.ylabel("Proporción de Resultados")
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.savefig(os.path.join(PLOT_DIR, "ventaja_localia_confederacion.png"), dpi=300, bbox_inches='tight')
plt.show()

### 5. Distribución Temporal y Top Selecciones

Graficamos la cantidad de partidos por año y los equipos con mayor volumen de encuentros.

In [ ]:
df['year'] = df['date'].dt.year
plt.figure(figsize=(10, 5))
sns.histplot(data=df, x='year', bins=30, kde=True, color='skyblue')
plt.title("Distribución Temporal de Partidos por Año")
plt.xlabel("Año")
plt.ylabel("Cantidad de Partidos")
plt.savefig(os.path.join(PLOT_DIR, "distribucion_temporal.png"), dpi=300, bbox_inches='tight')
plt.show()

top_20 = pd.concat([df['home_team'], df['away_team']]).value_counts().head(20)
plt.figure(figsize=(12, 6))
sns.barplot(x=top_20.values, y=top_20.index, palette='mako', hue=top_20.index, legend=False)
plt.title("Top 20 Selecciones con Más Partidos en el Dataset")
plt.xlabel("Cantidad de Partidos")
plt.ylabel("Selección")
plt.savefig(os.path.join(PLOT_DIR, "top_20_selecciones.png"), dpi=300, bbox_inches='tight')
plt.show()

### 6. Análisis de Sesgos del Dataset

- **Sesgo Temporal:** Existe una cantidad drásticamente superior de partidos registrados en las últimas décadas en comparación con los años iniciales. Esto se debe a la profesionalización y la creación de confederaciones, lo que puede influir en que el modelo asigne pesos desproporcionados a rachas recientes.
- **Ventaja de Localía:** En todas las confederaciones se observa una notable ventaja de jugar en casa, con la victoria local superando consistentemente el 40-50% de probabilidad en partidos no neutrales. CONMEBOL y CAF muestran tasas de victoria local particularmente altas.
- **Desequilibrio del Target:** Hay más victorias locales (`result=2`) que empates (`1`) o victorias visitantes (`0`). Los modelos ingenuos podrían sobrepredecir la localía.
- **Selecciones Frecuentes:** Hay un puñado de selecciones con más de 700 partidos registrados, mientras que algunas naciones más pequeñas solo tienen decenas. Esto genera diferencias en la madurez y estabilidad de las calificaciones ELO.

### 7. Particionado y Exportación de Features

Dividimos las variables de entrada según las fechas para entrenamiento (hasta 2018), validación (2019-2021) y test (2022-2024).

In [ ]:
feature_cols = [
    'elo_home', 'elo_away', 'elo_diff',
    'home_wins_5', 'home_draws_5', 'home_losses_5', 'home_goals_scored_5', 'home_goals_conceded_5', 'home_wins_10', 'home_wins_20',
    'away_wins_5', 'away_draws_5', 'away_losses_5', 'away_goals_scored_5', 'away_goals_conceded_5', 'away_wins_10', 'away_wins_20',
    'h2h_home_wins', 'h2h_draws', 'h2h_away_wins', 'h2h_home_goals_avg', 'h2h_away_goals_avg',
    'is_neutral', 'tournament_weight', 'phase_encoded', 'year', 'result'
]

train_df = df[df['year'] <= 2018][feature_cols]
val_df = df[(df['year'] >= 2019) & (df['year'] <= 2021)][feature_cols]
test_df = df[(df['year'] >= 2022) & (df['year'] <= 2024)][feature_cols]

# Imputar NaNs por precaución (mediana de train)
for col in feature_cols:
    if col != 'result':
        median_val = train_df[col].median()
        train_df[col] = train_df[col].fillna(median_val)
        val_df[col] = val_df[col].fillna(median_val)
        test_df[col] = test_df[col].fillna(median_val)

train_df.to_csv(os.path.join(PROCESSED_DIR, "features_train.csv"), index=False)
val_df.to_csv(os.path.join(PROCESSED_DIR, "features_val.csv"), index=False)
test_df.to_csv(os.path.join(PROCESSED_DIR, "features_test.csv"), index=False)

print(f"Particiones exportadas correctamente en {PROCESSED_DIR}:")
print(f"- features_train.csv: {train_df.shape}")
print(f"- features_val.csv: {val_df.shape}")
print(f"- features_test.csv: {test_df.shape}")